# Describe Data

Describe Data focus on examine the data as the data is placed without applying any modification, looking for surface properties such as data format, number of records and another field identities.

## Load Dataset

In [1]:
import pandas as pd
from pathlib import Path
from omegaconf import OmegaConf
from olist_ml.config import OlistConfig
from sqlalchemy import create_engine
root_path = project_root=Path.cwd().parent.parent
cfg = OmegaConf.load("../../conf/config.yaml")

raw_data_path = root_path / cfg.paths.raw_data
olist_sqlite_path = raw_data_path / cfg.files.olist_sqlite
db_file = f"sqlite:///{olist_sqlite_path}"
engine = create_engine(db_file)
orders = pd.read_sql("SELECT * FROM orders;", engine)
order_items = pd.read_sql("SELECT * FROM order_items;", engine)
products = pd.read_sql("SELECT * FROM products;", engine)
customers = pd.read_sql("SELECT * FROM customers;", engine)
product_category_name_translation = pd.read_sql("SELECT * FROM product_category_name_translation;", engine)
order_payments = pd.read_sql("SELECT * FROM order_payments;", engine)
order_reviews = pd.read_sql("SELECT * FROM order_reviews;", engine)
sellers = pd.read_sql("SELECT * FROM sellers;", engine)
geolocation = pd.read_sql("SELECT * FROM geolocation;", engine)
leads_closed = pd.read_sql("SELECT * FROM leads_closed;", engine)
leads_qualified = pd.read_sql("SELECT * FROM leads_qualified;", engine)

Olist Dataset correspond to an sqlite file, but it would consider every table on the database like an dataset. So next every Olist Dataset will be described.

1. Orders
2. Order Items
3. Products
4. Customers
5. Product Category Name Translation
6. Order Payments
7. Order Reviews
8. Sellers
9. Geolocation
10. Leads Closed
11. Leads Qualified

## Datasets

### 1. Orders

In [2]:
orders.shape, orders.columns, orders.isna().sum()

((99441, 8),
 Index(['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp',
        'order_approved_at', 'order_delivered_carrier_date',
        'order_delivered_customer_date', 'order_estimated_delivery_date'],
       dtype='object'),
 order_id                            0
 customer_id                         0
 order_status                        0
 order_purchase_timestamp            0
 order_approved_at                 160
 order_delivered_carrier_date     1783
 order_delivered_customer_date    2965
 order_estimated_delivery_date       0
 dtype: int64)

There are 99441 orders, an order contain 8 attributes.
- `order_id`, `customer_id`: `order_id` is the primary key and `customer_id` is a foreing key from the customers dataset.
- `order_status`: The step or stage in which the order is.
- `order_purchase_timestamp`: The date when the order place at.
- `order_approved_at`, `order_delivered_carrier_date`, `order_delivered_customer_date`, `order_estimated_delivery_date`: Dates related to the order status probably for logistical pourposes.

Some attributes have missing values, `order_approved_at` (160), `order_delivered_carrier_date` (1783), `order_delivered_customer_date` (2965).

In [3]:
orders_tmp = orders.copy()
columns = [
    "order_purchase_timestamp", 
    "order_approved_at", 
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"]
for col in columns:
    orders_tmp[col] = pd.to_datetime(orders_tmp[col])
orders.dtypes, orders_tmp.dtypes

(order_id                         object
 customer_id                      object
 order_status                     object
 order_purchase_timestamp         object
 order_approved_at                object
 order_delivered_carrier_date     object
 order_delivered_customer_date    object
 order_estimated_delivery_date    object
 dtype: object,
 order_id                                 object
 customer_id                              object
 order_status                             object
 order_purchase_timestamp         datetime64[ns]
 order_approved_at                datetime64[ns]
 order_delivered_carrier_date     datetime64[ns]
 order_delivered_customer_date    datetime64[ns]
 order_estimated_delivery_date    datetime64[ns]
 dtype: object)

In general dates are stored as objects data types, but date related data types are much more convinient for the data exploration and analysis. So temporaly `orders` will be parsed to a datetime type.

In [4]:
orders_tmp["order_status"].unique(), orders_tmp["order_status"].describe(), orders_tmp[[
    "order_purchase_timestamp", 
    "order_approved_at", 
    "order_delivered_carrier_date", 
    "order_delivered_customer_date", 
    "order_estimated_delivery_date"
]].agg(["min", "max"])

(array(['delivered', 'invoiced', 'shipped', 'processing', 'unavailable',
        'canceled', 'created', 'approved'], dtype=object),
 count         99441
 unique            8
 top       delivered
 freq          96478
 Name: order_status, dtype: object,
     order_purchase_timestamp   order_approved_at order_delivered_carrier_date  \
 min      2016-09-04 21:15:19 2016-09-15 12:16:38          2016-10-08 10:34:01   
 max      2018-10-17 17:30:18 2018-09-03 17:40:06          2018-09-11 19:48:28   
 
     order_delivered_customer_date order_estimated_delivery_date  
 min           2016-10-11 13:46:32                    2016-09-30  
 max           2018-10-17 13:22:46                    2018-11-12  )

- There are 8 order status, `delivered`, `invoiced`, `shipped`, `processing`, `unavailable`, `canceled`, `created` and `approved`.
- `delivered` order status is the most frequently, so it is more probably to take an `delivered` order instead of another order on different step.
- date-related columns show an acceptable range of time (that comes from september 2016 to october 2018) to use the data and identify patterns considering dates.

### 2. Order Items

In [5]:
order_items.shape, order_items.columns

((112650, 7),
 Index(['order_id', 'order_item_id', 'product_id', 'seller_id',
        'shipping_limit_date', 'price', 'freight_value'],
       dtype='object'))

There are 112650 order items, an order item contains 7 attributes.
- `order_id`, `order_item_id`, `product_id`, `seller_id`: `order_item_id` is a sequential number assigned to an order item to distinct this with another order item, so values are not uniques and can repeat in order items of different orders. `order_id`, `product_id`, `seller_id` are foreing keys from orders, products and sellers datasets.
- `shipping_limit_date`: The deadline in which the seller must to provide the package to the shipping company.
- `price`, `freight_value`: `price` refers to the order item cost and `freight_value` refers to the item shipping cost (considering measures and weight).
- The Kaggle dataset documentation mention that the total order value is the sum of the total of the order items values `price` and the total of the freight values `freight_value`.
- The Kaggle dataset documentation do not mention explicity the type of monentary value, so it is infered from the e-commerce location Brazil, so it is supposed that use BRL Brazilian Real.

In [6]:
order_items_tmp = order_items.copy()
order_items_tmp["shipping_limit_date"] = pd.to_datetime(order_items_tmp["shipping_limit_date"])
order_items.dtypes, order_items_tmp.dtypes

(order_id                object
 order_item_id            int64
 product_id              object
 seller_id               object
 shipping_limit_date     object
 price                  float64
 freight_value          float64
 dtype: object,
 order_id                       object
 order_item_id                   int64
 product_id                     object
 seller_id                      object
 shipping_limit_date    datetime64[ns]
 price                         float64
 freight_value                 float64
 dtype: object)

Similarly with orders dataset, `shipping_limit_date` should be convinient to stored it with a date related data type.

In [7]:
order_items_tmp[["price", "freight_value", "shipping_limit_date"]].agg(["min", "max"])

,price,freight_value,shipping_limit_date
min,0.85,0.00,2016-09-19 00:15:34
max,6735.00,409.68,2020-04-09 22:35:08


- `price` shows a maximun value of 6735 BRL, which is about ~2108 USD (using a change of 0.313 USD/BRL in 2017) which is reasonably into high value products of a ecommerce.
- `freight_value` has a maximun value of 409 BRL, which is about ~128 USD (using a change of 0.313 USD/BRL in 2017) which is reasonably between heavy or complex products  of a ecommerce.
- `shipping_limit_date` exceed the typical range of years of order purchase 2016-2018, so in some cases, the deadline for sellers were more flexible at the point to consider 2020.

### 3. Products

In [8]:
products.shape, products.columns, products.isna().sum()

((32951, 9),
 Index(['product_id', 'product_category_name', 'product_name_lenght',
        'product_description_lenght', 'product_photos_qty', 'product_weight_g',
        'product_length_cm', 'product_height_cm', 'product_width_cm'],
       dtype='object'),
 product_id                      0
 product_category_name         610
 product_name_lenght           610
 product_description_lenght    610
 product_photos_qty            610
 product_weight_g                2
 product_length_cm               2
 product_height_cm               2
 product_width_cm                2
 dtype: int64)

There are 32951 products, a product contain 9 attributes.
- `product_id`, `product_category_name`: `product_id` is the primary key, `product_category_name` is the foreing key from the product category name translation dataset.
- `product_name_lenght`, `product_description_lenght`, `product_photos_qty`: Information about the product details. `product_name_lenght` and `product_description_lenght` contain typos, it should be `product_name_length` and `product_description_length`.
- `product_weight_g`, `product_length_cm`, `product_height_cm`, `product_width_cm`: Measures and weight of the product in grams and centimeters.

Almost all of the attributes have missing values, `product_category_name` (610), `product_name_lenght` (610), `product_description_lenght` (610), `product_photos_qty` (610), `product_weight_g` (2), `product_length_cm` (2), `product_height_cm` (2), `product_width_cm` (2).

In [9]:
products.dtypes

product_id                     object
product_category_name          object
product_name_lenght           float64
product_description_lenght    float64
product_photos_qty            float64
product_weight_g              float64
product_length_cm             float64
product_height_cm             float64
product_width_cm              float64
dtype: object

Although `product_name_lenght`, `product_description_lenght`, `product_weight_g`, `product_length_cm`, `product_height_cm`, `product_width_cm` are stored in float data type, they contain discrete values that in very special situation may introduce noise.

In [10]:
sorted(products["product_category_name"].dropna().unique()), products["product_category_name"].describe()

(['agro_industria_e_comercio',
  'alimentos',
  'alimentos_bebidas',
  'artes',
  'artes_e_artesanato',
  'artigos_de_festas',
  'artigos_de_natal',
  'audio',
  'automotivo',
  'bebes',
  'bebidas',
  'beleza_saude',
  'brinquedos',
  'cama_mesa_banho',
  'casa_conforto',
  'casa_conforto_2',
  'casa_construcao',
  'cds_dvds_musicais',
  'cine_foto',
  'climatizacao',
  'consoles_games',
  'construcao_ferramentas_construcao',
  'construcao_ferramentas_ferramentas',
  'construcao_ferramentas_iluminacao',
  'construcao_ferramentas_jardim',
  'construcao_ferramentas_seguranca',
  'cool_stuff',
  'dvds_blu_ray',
  'eletrodomesticos',
  'eletrodomesticos_2',
  'eletronicos',
  'eletroportateis',
  'esporte_lazer',
  'fashion_bolsas_e_acessorios',
  'fashion_calcados',
  'fashion_esporte',
  'fashion_roupa_feminina',
  'fashion_roupa_infanto_juvenil',
  'fashion_roupa_masculina',
  'fashion_underwear_e_moda_praia',
  'ferramentas_jardim',
  'flores',
  'fraldas_higiene',
  'industria_comerc

- There are 73 product categories.
- Some of the division are not very clear, e.j. `eletrodomesticos`, `eletrodomesticos_2` and `casa_conforto`, `casa_conforto_2`.
- `cama_mesa_baho` is the category is the most frequent with just 3029 products.

### 4. Customers

In [11]:
customers.shape, customers.columns

((99441, 5),
 Index(['customer_id', 'customer_unique_id', 'customer_zip_code_prefix',
        'customer_city', 'customer_state'],
       dtype='object'))

There are 99441 times that customers made orders, an instance of customer have 5 attributes.
- `customer_id`, `customer_unique_id`: `customer_id` is the primary key representing an instance of a customer made a order in a point of time, `customer_unique_id` is a foreing key of an unknowed dataset of unique customers, represent the unique or final customer, `customer_zip_code_prefix` is a foreing key of geolocation dataset.
- `customer_zip_code_prefix`, `customer_city`, `customer_state`: Location of the customer in the instance that they made an order.

In [12]:
customers[['customer_city', 'customer_state']].describe()

,customer_city,customer_state
count,99441,99441
unique,4119,27
top,sao paulo,SP
freq,15540,41746


The municipality of Sao Paulo and State of Sao Paulo are considerable prefered for the customers at the moment to made an order. So at least the middle of times that customers made orders were from State of Sao Paulo.

### 5. Product Category Name Translation

In [13]:
product_category_name_translation.shape, product_category_name_translation.columns

((71, 2),
 Index(['product_category_name', 'product_category_name_english'], dtype='object'))

There are 71 name translation of product category, a name translation just contain the translation in English.
- `product_category_name`: Is the primary key.
- The previous product section showed 73 unique product category names, so there are 2 untracked categories for name translation.

In [14]:
sorted(product_category_name_translation["product_category_name_english"].unique())

['agro_industry_and_commerce',
 'air_conditioning',
 'art',
 'arts_and_craftmanship',
 'audio',
 'auto',
 'baby',
 'bed_bath_table',
 'books_general_interest',
 'books_imported',
 'books_technical',
 'cds_dvds_musicals',
 'christmas_supplies',
 'cine_photo',
 'computers',
 'computers_accessories',
 'consoles_games',
 'construction_tools_construction',
 'construction_tools_lights',
 'construction_tools_safety',
 'cool_stuff',
 'costruction_tools_garden',
 'costruction_tools_tools',
 'diapers_and_hygiene',
 'drinks',
 'dvds_blu_ray',
 'electronics',
 'fashio_female_clothing',
 'fashion_bags_accessories',
 'fashion_childrens_clothes',
 'fashion_male_clothing',
 'fashion_shoes',
 'fashion_sport',
 'fashion_underwear_beach',
 'fixed_telephony',
 'flowers',
 'food',
 'food_drink',
 'furniture_bedroom',
 'furniture_decor',
 'furniture_living_room',
 'furniture_mattress_and_upholstery',
 'garden_tools',
 'health_beauty',
 'home_appliances',
 'home_appliances_2',
 'home_comfort_2',
 'home_confo

`product_category_name_english` have the same problem showed in product section, some names are not very descriptives, e.j. `home_appliances`, `home_appliances_2` and `home_comfort_2`, `home_confort`.

### 6. Order Payments

In [15]:
order_payments.shape, order_payments.columns

((103886, 5),
 Index(['order_id', 'payment_sequential', 'payment_type',
        'payment_installments', 'payment_value'],
       dtype='object'))

There are 103886 order payments, an order payment contain 5 attributes.
- `order_id`, `payment_sequential`: `order_id` is a foreign key from orders dataset, `payment_sequential` represent sequential number of each payment for that order which can be splitted by unknowed rules of the system.
- `payment_type`: Is the payment method.
- `payment_installments`: Represent the payment installments or small portions to pay for.
- `payment_value`: Is the transaction value, similarly in order item section, due to the ecommerce location it is infered to be in BRL Brazilian Real.

In [16]:
order_payments["payment_value"].agg(["min", "max"]), len(order_payments[order_payments["payment_value"] == 0])

(min        0.00
 max    13664.08
 Name: payment_value, dtype: float64,
 9)

`payment_value` has a minimun of 0 BRL, but it just happens 9 times, so may be something marginal.

In [17]:
order_payments["payment_type"].unique(), order_payments["payment_type"].describe()

(array(['credit_card', 'boleto', 'voucher', 'debit_card', 'not_defined'],
       dtype=object),
 count          103886
 unique              5
 top       credit_card
 freq            76795
 Name: payment_type, dtype: object)

There are 5 payment types, `credit_card`, `boleto`, `voucher`, `debit_card`, `not_defined`.

Then the most frequently method used is `credit_card` with 76795 order payments.

### 7. Order Reviews

In [18]:
order_reviews.shape, order_reviews.columns, order_reviews.isna().sum()

((99224, 7),
 Index(['review_id', 'order_id', 'review_score', 'review_comment_title',
        'review_comment_message', 'review_creation_date',
        'review_answer_timestamp'],
       dtype='object'),
 review_id                      0
 order_id                       0
 review_score                   0
 review_comment_title       87656
 review_comment_message     58247
 review_creation_date           0
 review_answer_timestamp        0
 dtype: int64)

There are 99224 order reviews, an order review contain 7 attributes.
- `review_id`, `order_id`: `review_id` is the primary key, `order_id` is a foreign key from the orders dataset.
- `review_score`: review score in which 5 is the better.
- `review_comment_title`, `review_comment_message`: customer review title and body.
- `review_creation_date`, `review_answer_timestamp`: when the survey was sent and answered.

There are considerable missing values `review_comment_title` (87656) and `review_comment_message` (58247).

In [19]:
order_reviews.dtypes
order_reviews_tmp = order_reviews.copy()
columns = ["review_creation_date", "review_answer_timestamp"]
for col in columns:
    order_reviews_tmp[col] = pd.to_datetime(order_reviews_tmp[col])
order_reviews.dtypes, order_reviews_tmp.dtypes

(review_id                  object
 order_id                   object
 review_score                int64
 review_comment_title       object
 review_comment_message     object
 review_creation_date       object
 review_answer_timestamp    object
 dtype: object,
 review_id                          object
 order_id                           object
 review_score                        int64
 review_comment_title               object
 review_comment_message             object
 review_creation_date       datetime64[ns]
 review_answer_timestamp    datetime64[ns]
 dtype: object)

Similar to previous datasets, `review_creation_date` and `review_answer_timestamp` should be stored in a date-related data type.

In [20]:
order_reviews_tmp["review_score"].value_counts()

review_score
5    57328
4    19142
1    11424
3     8179
2     3151
Name: count, dtype: int64

It is more probably to take a positive score review than a negative score.

### 8. Sellers

In [21]:
sellers.shape, sellers.columns

((3095, 4),
 Index(['seller_id', 'seller_zip_code_prefix', 'seller_city', 'seller_state'], dtype='object'))

There are 3095 sellers, a seller contain 4 attributes.
- `seller_id`, `seller_zip_code_prefix`: `seller_id` is the primary key, `seller_zip_code_prefix` is a foreign key from geolocation dataset.
- `seller_city`, `seller_state`: Represent the seller location.

In [22]:
sellers[["seller_city", "seller_state"]].describe()

,seller_city,seller_state
count,3095,3095
unique,611,23
top,sao paulo,SP
freq,694,1849


The most preferred seller location is municipality of Sao Paulo and Sao Paulo State.

### 9. Geolocation

In [23]:
geolocation.shape, geolocation.columns

((1000163, 5),
 Index(['geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng',
        'geolocation_city', 'geolocation_state'],
       dtype='object'))

There are more than 1000000 geolocations, a geolocation contain 5 attributes.
- `geolocation_zip_code_prefix`: Is the primary key.
- `geolocation_lat`, `geolocation_lng`: Represent latitude and longitude coordinates.
- `geolocation_city`, `geolocation_state`: Represent city and state.

In [24]:
geolocation[["geolocation_city", "geolocation_state"]].describe()

,geolocation_city,geolocation_state
count,1000163,1000163
unique,8011,27
top,sao paulo,SP
freq,135800,404268


The location most frequently are the municipality of Sao Paulo and Sao Paulo state.

### 10. Leads Closed

In [25]:
leads_closed.shape, leads_closed.columns, leads_closed.isna().sum()

((842, 14),
 Index(['mql_id', 'seller_id', 'sdr_id', 'sr_id', 'won_date',
        'business_segment', 'lead_type', 'lead_behaviour_profile',
        'has_company', 'has_gtin', 'average_stock', 'business_type',
        'declared_product_catalog_size', 'declared_monthly_revenue'],
       dtype='object'),
 mql_id                             0
 seller_id                          0
 sdr_id                             0
 sr_id                              0
 won_date                           0
 business_segment                   1
 lead_type                          6
 lead_behaviour_profile           177
 has_company                      779
 has_gtin                         778
 average_stock                    776
 business_type                     10
 declared_product_catalog_size    773
 declared_monthly_revenue           0
 dtype: int64)

There are 842 leads closed, a lead closed contain 14 attributes.

- `mql_id`, `seller_id`, `sdr_id`, `sr_id`: `mql_id` is a foreign key from the leads qualified dataset, `seller_id` is a foreign key from the sellers dataset, `sdr_id` is a foreign key for the sales development representative, `sr_id` is a foreign key for the representative.
- `won_date`: Date when the deal was closed.
- `business_segment`, `business_type`: `business_segment` is the market segment of the lead, `business_type` is the type of business or company model of the lead.
- `lead_type`, `lead_behaviour_profile`: `lead_type` is a classification of the lead and `lead_behaviour_profile` is the type of behaviour profile according to the sales development representative.
- `has_company`, `has_gtin`: Whether the lead has company documentation and use barcode for their products.
- `average_stock`: Average stock declared by the lead for products to be offered.
- `declared_product_catalog_size`, `declared_monthly_revenue`: `declared_product_catalog_size` is a declaration number of products the lead intends to list in the catalog, `declared_monthly revenue` is the estimated monthly revenue expected from the lead.

There are a considerable number of missing values, `business_segment` (1), `lead_type` (6), `lead_behaviour_profile` (177), `has_company` (779), `has_gtin` (778), `average_stock` (776), `business_type` (10), `declared_product_catalog_size` (773).

In [26]:
leads_closed_tmp = leads_closed.copy()
leads_closed_tmp["won_date"] = pd.to_datetime(leads_closed_tmp["won_date"])
leads_closed.dtypes, leads_closed_tmp.dtypes

(mql_id                            object
 seller_id                         object
 sdr_id                            object
 sr_id                             object
 won_date                          object
 business_segment                  object
 lead_type                         object
 lead_behaviour_profile            object
 has_company                      float64
 has_gtin                         float64
 average_stock                     object
 business_type                     object
 declared_product_catalog_size    float64
 declared_monthly_revenue         float64
 dtype: object,
 mql_id                                   object
 seller_id                                object
 sdr_id                                   object
 sr_id                                    object
 won_date                         datetime64[ns]
 business_segment                         object
 lead_type                                object
 lead_behaviour_profile                   object
 has

Similar to previous sections date related columns such as `won_date` should be stored as a datetime data type.

### 11. Leads Qualified

In [27]:
leads_qualified.shape, leads_qualified.columns, leads_qualified.isna().sum()

((8000, 4),
 Index(['mql_id', 'first_contact_date', 'landing_page_id', 'origin'], dtype='object'),
 mql_id                 0
 first_contact_date     0
 landing_page_id        0
 origin                60
 dtype: int64)

- `mql_id`, `landing_page_id`: `mql_id` is the primary key, `landing_page_id` is an identifier where the lead was acquired.
- `first_contact_date`: Date of the first contact solicitation of the lead.
- `origin`: Type of media were the lead was acquired.

There are some missing values, `origin` (60).

In [28]:
leads_qualified_tmp = leads_qualified.copy()
leads_qualified_tmp["first_contact_date"] = pd.to_datetime(leads_qualified_tmp["first_contact_date"])
leads_qualified.dtypes, leads_qualified_tmp.dtypes

(mql_id                object
 first_contact_date    object
 landing_page_id       object
 origin                object
 dtype: object,
 mql_id                        object
 first_contact_date    datetime64[ns]
 landing_page_id               object
 origin                        object
 dtype: object)

Similar that other sections, the `first_contanct_date` should be stored in a date related data type.